In [1]:
# CRITICAL: Set environment variables before any imports
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'  # Disable xet transfer that causes issues
os.environ['HF_HUB_DISABLE_XET'] = '1'

# Also monkey-patch tqdm before any imports
import sys

# Create a dummy tqdm that does nothing
class DummyTqdm:
    def __init__(self, *args, **kwargs):
        self.iterable = args[0] if args else kwargs.get('iterable', [])
    def __iter__(self):
        return iter(self.iterable if self.iterable else [])
    def __enter__(self):
        return self
    def __exit__(self, *args):
        pass
    def update(self, *args, **kwargs):
        pass
    def close(self):
        pass
    def set_description(self, *args, **kwargs):
        pass
    @staticmethod
    def pandas(*args, **kwargs):
        pass

# Pre-emptively set up tqdm module
import tqdm.std
tqdm.std.tqdm = DummyTqdm

print("Environment configured to disable progress bars")

Environment configured to disable progress bars


In [2]:
# Set working directory
os.chdir('/net/scratch2/smallyan/filter_eval')
sys.path.insert(0, '/net/scratch2/smallyan/filter_eval')
print(f"Working directory: {os.getcwd()}")

Working directory: /net/scratch2/smallyan/filter_eval


In [3]:
# Now try loading the model
import torch
import transformers

print(f"{torch.__version__=}, {torch.version.cuda=}")
print(f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}")
if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name()=}")
print(f"{transformers.__version__=}")

torch.__version__='2.7.1+cu118', torch.version.cuda='11.8'
torch.cuda.is_available()=True, torch.cuda.device_count()=1
torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
transformers.__version__='4.57.3'


In [4]:
# Try to load the model with local_files_only=True to avoid download issues
from src.models import ModelandTokenizer

model_key = "meta-llama/Llama-3.3-70B-Instruct"

try:
    mt = ModelandTokenizer(
        model_key=model_key,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
        local_files_only=True,  # Try local cache first
    )
    print("Model loaded from local cache!")
except Exception as e:
    print(f"Local cache failed: {e}")
    print("\nTrying to load without local_files_only...")

meta-llama/Llama-3.3-70B-Instruct not found in /net/projects/chai-lab/shared_models
If not found in cache, model will be downloaded from HuggingFace to cache directory


`torch_dtype` is deprecated! Use `dtype` instead!


`torch_dtype` is deprecated! Use `dtype` instead!


Local cache failed: meta-llama/Llama-3.3-70B-Instruct does not appear to have files named ('model-00001-of-00030.safetensors', 'model-00002-of-00030.safetensors', 'model-00003-of-00030.safetensors', 'model-00004-of-00030.safetensors', 'model-00005-of-00030.safetensors', 'model-00006-of-00030.safetensors', 'model-00007-of-00030.safetensors', 'model-00008-of-00030.safetensors', 'model-00009-of-00030.safetensors', 'model-00010-of-00030.safetensors', 'model-00011-of-00030.safetensors', 'model-00012-of-00030.safetensors', 'model-00013-of-00030.safetensors', 'model-00014-of-00030.safetensors', 'model-00015-of-00030.safetensors', 'model-00016-of-00030.safetensors', 'model-00017-of-00030.safetensors', 'model-00018-of-00030.safetensors', 'model-00019-of-00030.safetensors', 'model-00020-of-00030.safetensors', 'model-00021-of-00030.safetensors', 'model-00022-of-00030.safetensors', 'model-00023-of-00030.safetensors', 'model-00024-of-00030.safetensors', 'model-00025-of-00030.safetensors', 'model-00

In [5]:
# Try loading without local_files_only but with our patched environment
# The model needs to be downloaded

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("Model loaded successfully!")

meta-llama/Llama-3.3-70B-Instruct not found in /net/projects/chai-lab/shared_models
If not found in cache, model will be downloaded from HuggingFace to cache directory


AttributeError: type object 'tqdm' has no attribute 'get_lock'

In [6]:
# Given the model loading issues with tqdm/ipywidgets in this environment,
# let me take a different approach: analyze code correctness statically and 
# test what we can without the full model

# First, let's analyze the source code structure and check for obvious issues
import json

# Read and parse the demo notebook
with open('/net/scratch2/smallyan/filter_eval/demo.ipynb', 'r') as f:
    notebook = json.load(f)

print("=== DEMO.IPYNB CODE ANALYSIS ===\n")

# Extract all code cells
code_cells = []
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'code':
        source = ''.join(cell['source'])
        outputs = cell.get('outputs', [])
        has_error_output = any(out.get('output_type') == 'error' for out in outputs)
        code_cells.append({
            'cell_index': len(code_cells),
            'notebook_index': i,
            'source': source,
            'has_outputs': len(outputs) > 0,
            'has_error': has_error_output
        })
        
print(f"Total code cells: {len(code_cells)}")
for j, cell in enumerate(code_cells):
    print(f"\n--- Cell {j} (notebook idx {cell['notebook_index']}) ---")
    lines = cell['source'].split('\n')[:5]
    for line in lines:
        print(f"  {line[:80]}")
    if len(cell['source'].split('\n')) > 5:
        print("  ...")
    print(f"  [Has outputs: {cell['has_outputs']}, Has error: {cell['has_error']}]")

=== DEMO.IPYNB CODE ANALYSIS ===

Total code cells: 16

--- Cell 0 (notebook idx 0) ---
  %load_ext autoreload
  %autoreload 2
  [Has outputs: False, Has error: False]

--- Cell 1 (notebook idx 1) ---
  import torch
  import transformers
  from src.models import ModelandTokenizer
  
  print(f"{torch.__version__=}, {torch.version.cuda=}")
  ...
  [Has outputs: True, Has error: False]

--- Cell 2 (notebook idx 2) ---
  # select one of the filter heads
  if model_key == "meta-llama/Llama-3.3-70B-Instruct":
      layer_idx, head_idx = 35, 19
  elif model_key == "google/gemma-2-27b-it":
      layer_idx, head_idx = 29, 3
  ...
  [Has outputs: False, Has error: False]

--- Cell 3 (notebook idx 4) ---
  from src.selection.data import SelectOneTask
  from typing import Literal
  import os
  # from src.utils import env_utils # you should create the env.yml file as per in
  
  ...
  [Has outputs: True, Has error: False]

--- Cell 4 (notebook idx 5) ---
  sample = select_task.get_random_sample(
  

In [7]:
# The notebook shows that all cells (except last two empty cells) were executed successfully 
# in the original run with outputs and no errors.
# 
# Given the model loading issues in this MCP environment, I'll:
# 1. Test imports and code syntax validity
# 2. Check that the source modules exist and have correct functions
# 3. Analyze code correctness based on the plan/methodology

# Let's test imports systematically
import_tests = []

# Test 1: Core imports
test_name = "Core imports (torch, transformers)"
try:
    import torch
    import transformers
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 2: src.models
test_name = "src.models.ModelandTokenizer"
try:
    from src.models import ModelandTokenizer
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 3: src.selection.data
test_name = "src.selection.data imports"
try:
    from src.selection.data import SelectOneTask, get_counterfactual_samples_within_task, MCQify_sample
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 4: src.selection.functional
test_name = "src.selection.functional imports"
try:
    from src.selection.functional import verify_head_patterns, cache_q_projections
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 5: src.selection.utils
test_name = "src.selection.utils imports"
try:
    from src.selection.utils import get_first_token_id
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 6: src.tokens
test_name = "src.tokens.prepare_input"
try:
    from src.tokens import prepare_input
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

# Test 7: src.functional
test_name = "src.functional imports"
try:
    from src.functional import interpret_logits, PatchSpec
    import_tests.append((test_name, "PASS", ""))
except Exception as e:
    import_tests.append((test_name, "FAIL", str(e)))

print("=== IMPORT TESTS ===")
for name, status, error in import_tests:
    print(f"{status}: {name}")
    if error:
        print(f"  Error: {error[:100]}")

=== IMPORT TESTS ===
PASS: Core imports (torch, transformers)
PASS: src.models.ModelandTokenizer
PASS: src.selection.data imports
PASS: src.selection.functional imports
PASS: src.selection.utils imports
PASS: src.tokens.prepare_input
PASS: src.functional imports


In [8]:
# Test data loading (doesn't require model)
test_name = "Load SelectOneTask data"
try:
    select_task = SelectOneTask.load(
        path=os.path.join("data_save", "selection", "objects.json")
    )
    print(f"PASS: {test_name}")
    print(f"  Task name: {select_task.name if hasattr(select_task, 'name') else 'N/A'}")
    print(f"  Categories: {select_task.categories[:5] if hasattr(select_task, 'categories') else 'N/A'}...")
except Exception as e:
    print(f"FAIL: {test_name}")
    print(f"  Error: {e}")

['name', 'prompt_templates', 'odd_one_prompt_templates', 'order_prompt_templates', 'count_prompt_templates', 'yes_no_prompt_templates', 'first_item_in_cat_prompt_templates', 'last_item_in_cat_prompt_templates', 'categories', 'exclude_categories']
PASS: Load SelectOneTask data
  Task name: N/A
  Categories: ['fruit', 'vehicle', 'furniture', 'animal', 'music instrument']...


In [9]:
# Now let's try loading a smaller model to test the full pipeline
# Let's use a smaller model that might work with limited GPU memory

# First, let's check if we can load any model at all by trying with 
# a simpler approach - loading just the tokenizer first
print("Testing tokenizer loading...")
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.3-70B-Instruct")
    print(f"PASS: Tokenizer loaded successfully")
    print(f"  Vocab size: {tokenizer.vocab_size}")
except Exception as e:
    print(f"FAIL: {e}")

Testing tokenizer loading...


PASS: Tokenizer loaded successfully
  Vocab size: 128000


In [10]:
# Try to patch the huggingface_hub to use simpler download
import huggingface_hub
import huggingface_hub._snapshot_download as snap_download

# Patch the snapshot_download to use single-threaded downloads
original_snapshot = huggingface_hub.snapshot_download

def patched_snapshot(*args, **kwargs):
    kwargs['max_workers'] = 1  # Force single-threaded
    return original_snapshot(*args, **kwargs)

huggingface_hub.snapshot_download = patched_snapshot

print("Patched snapshot_download to use single worker")

Patched snapshot_download to use single worker


In [11]:
# Try loading the model again
print("Attempting to load model...")
mt = ModelandTokenizer(
    model_key="meta-llama/Llama-3.3-70B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("Model loaded successfully!")

meta-llama/Llama-3.3-70B-Instruct not found in /net/projects/chai-lab/shared_models
If not found in cache, model will be downloaded from HuggingFace to cache directory


Attempting to load model...


AttributeError: type object 'tqdm' has no attribute 'get_lock'